# Practical PyTorch · II · Part 8 — Companion Notebook

This goes with **"When Not to Fine-Tune"**. Run the cells top to bottom (Shift+Enter).

The point of this notebook is to *feel* the cheaper alternatives to fine-tuning — few-shot prompting and a tiny retrieval sketch — so you reach for them first. Nothing here trains a model; it all runs in seconds on CPU.

In [ ]:
!pip install -q transformers

## 1. Few-shot prompting: teach the task in the prompt

Before fine-tuning anything, try *showing* the model what you want. Here we build a few-shot prompt for a support-ticket classifier — a few labeled examples, then a new ticket to label. No dataset, no training run, no GPU.

We'll use a small instruction-following model so it runs anywhere. The exact model matters less than the shape of the prompt.

In [ ]:
from transformers import pipeline

# A small, free model so this runs quickly on CPU.
generate = pipeline("text2text-generation", model="google/flan-t5-base")

prompt = """Classify each support ticket as BILLING, BUG, or FEATURE.

Ticket: "I was charged twice this month."
Label: BILLING

Ticket: "The export button does nothing when I click it."
Label: BUG

Ticket: "Could you add dark mode?"
Label: FEATURE

Ticket: "My invoice shows the wrong amount."
Label:"""

print(generate(prompt, max_new_tokens=5)[0]["generated_text"])

Three examples in the prompt specified the task, the label set, *and* the output shape. Try editing the examples or the final ticket and re-running — you just "retrained" the classifier in ten seconds, with no training at all.

Swap in your own categories below to feel how flexible this is.

In [ ]:
tickets = [
    "Why was I billed in USD instead of EUR?",
    "The app crashes when I upload a PDF.",
    "Please support exporting to CSV.",
]

header = prompt.rsplit("Ticket:", 1)[0]  # reuse the 3 examples above

for t in tickets:
    p = f'{header}Ticket: "{t}"\nLabel:'
    label = generate(p, max_new_tokens=5)[0]["generated_text"]
    print(f"{label:8}  <-  {t}")

## 2. RAG, the tiny version: retrieve then prompt

When prompting fails because the model simply *doesn't know your facts*, the fix usually isn't fine-tuning — it's **retrieval-augmented generation**: look the facts up at question time and paste them into the prompt.

Here's the whole idea in miniature. We have a tiny "knowledge base" of facts the model was never trained on. A real system uses vector search; we'll use crude keyword overlap so the *flow* is visible without extra libraries.

In [ ]:
# Facts the base model has no way of knowing — they're made up for this demo.
documents = [
    "Acme Cloud's Pro plan costs $49 per month and includes 2 TB of storage.",
    "The Acme support team is available Monday to Friday, 9am to 6pm UK time.",
    "Acme data centers are located in London, Frankfurt, and Virginia.",
    "Refunds are issued within 14 days of purchase, no questions asked.",
]

def search(docs, question, top_k=2):
    """Crude relevance: rank docs by shared words with the question."""
    q = set(question.lower().split())
    scored = sorted(docs, key=lambda d: len(q & set(d.lower().split())), reverse=True)
    return scored[:top_k]

question = "How much does the Pro plan cost and how much storage do I get?"
print("Retrieved:")
for d in search(documents, question):
    print(" -", d)

Now the second step: hand those retrieved facts to the model and ask it to answer **using only the provided context**. The model supplies the reasoning; the retrieval supplied the knowledge.

In [ ]:
def answer(question, documents):
    relevant = search(documents, question, top_k=2)          # 1. RETRIEVE
    context = "\n".join(relevant)
    prompt = (                                               # 2. GENERATE
        "Answer the question using only the context below.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )
    return generate(prompt, max_new_tokens=40)[0]["generated_text"]

print(answer("How much does the Pro plan cost?", documents))
print(answer("When can I reach support?", documents))
print(answer("Can I get a refund after 10 days?", documents))

The model answered facts it was never trained on — because we *gave* it the facts at question time. Edit a document (change the Pro plan price, say) and re-run: the answer updates instantly, with no retraining. That's the whole pitch of RAG over fine-tuning when the problem is missing knowledge.

A real system swaps the keyword `search` for a vector search over embeddings, which finds relevant chunks by *meaning* rather than shared words — but the two-step retrieve-then-prompt shape is exactly this.

## 3. The decision checklist

When you're tempted to fine-tune, walk down this ladder and stop at the first rung that works:

1. **Better prompting / few-shot** — show the model examples (cell 1). Push this further than feels necessary.
2. **RAG** — if the problem is *missing knowledge* rather than *missing skill* (cell 2). Retrieve the facts, paste them in.
3. **A bigger or newer off-the-shelf model** — the cheapest experiment in the building; often ends the project.
4. **Then** fine-tune — for consistent behavior, a narrow high-volume task, or a domain where you've truly hit the wall.

Rule of thumb: **fine-tune to change how the model behaves, not to tell it facts.** Facts are RAG's job; behavior is fine-tuning's. If you can't say which rung failed and why, you're not ready for the next one.